# Data Preprocessing Notebook
- This notebook contains essential preprocessing steps to prepare a dataset for ML models.

|  Field  |                       Details                       | Status |
| :-----: | :-------------------------------------------------: | :----: |
|   Name  |                     Mahboob Alam                    | Active |
|  Email  | [infoatdoor@gmail.com](mailto:infoatdoor@gmail.com) | Active |
| Project |                     Data Analysis                    | Active |


## 1. Loading Libraries

In [15]:
# Import necessary libraries
#import numpy as np
import matplotlib
import pandas as pd
import sqlite3   # if importing data from a SQLite database
import pdfplumber
#from scipy import stats
#from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os
import itertools

# settings for Better Data Display in Jupyter Notebook

pd.set_option('display.max_rows', None)         # Show all rows
pd.set_option('display.max_columns', None)      # Show all columns
pd.set_option('display.max_colwidth', None)     # Show full column content (even long text)
pd.set_option('display.expand_frame_repr', False)  # Prevent line-wrapping of DataFrames

# Enable seaborn style
sns.set_style("whitegrid")
matplotlib.use('module://positron.matplotlib_backend') 



## 2. Loading Data

#### 2.1 For CSV data

In [ ]:
# Load the dataset 
RAW_CSV_PATH = '../data/Titanic-Dataset.csv'  # Remember to update file name and use / instead of \ in the path
# df = pd.read_csv(RAW_CSV_PATH, low_memory=False)   # Use this line if  having memory issues just to ignore erorrs
df = pd.read_csv(RAW_CSV_PATH)
print(f"✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

#### 2.2 For Excel Data

In [ ]:
# Load the dataset
RAW_EXCEL_PATH = "../data/Titanic-Dataset.xlsx"

df = pd.read_excel(RAW_EXCEL_PATH)

print(f"✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

#### 2.3  For XML Data

In [ ]:
# Load the dataset
RAW_XML_PATH = "../data/Titanic-Dataset.xml"

df = pd.read_xml(RAW_XML_PATH)

print(f"✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

#### 2.4 For JSON data

In [ ]:
# Load the dataset
RAW_JSON_PATH = "../data/cocoon.json"

df = pd.read_json(RAW_JSON_PATH)

print(f"✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Display basic info
df.info()

#### 2.5 For local SQL Data

In [ ]:
# Load the dataset
DATABASE_PATH = "../data/shiori.sql"

conn = sqlite3.connect(DATABASE_PATH)

df = pd.read_sql("SELECT * FROM table_name", conn)   #Replace table_name with the actual table name in your database.

conn.close()



#### for Azure database

#### Step 1: Install the required package (run once in terminal)
pip install pyodbc

#### Step 2: Import libraries

In [16]:
import pyodbc # to connect azure database
import urllib
from sqlalchemy import create_engine

In [17]:
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 18 for SQL Server']


#### Step 3: Define your connection details

Replace the placeholders with your own values.

In [ ]:
server = "write servername here"
database = "write database name here"
username = "write username here"
password = "write password here"


# 2. Format the connection string for SQLAlchemy
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)
connection_uri = f"mssql+pyodbc:///?odbc_connect={params}"

# 3. Create the SQLAlchemy engine wrapper
engine = create_engine(connection_uri)

# 4. Define your structural metadata query
query = """
SELECT TABLE_SCHEMA, TABLE_NAME 
FROM INFORMATION_SCHEMA.TABLES 
WHERE TABLE_TYPE = 'BASE TABLE' 
ORDER BY TABLE_SCHEMA, TABLE_NAME;
"""

# 5. Read using the engine instead of raw conn
tables = pd.read_sql(query, engine)

# Display the tables DataFrame
print(tables)

Empty DataFrame
Columns: [TABLE_SCHEMA, TABLE_NAME]
Index: []


#### Step 4: Connect to the database

In [19]:
# Clean single-line connection string to prevent parsing errors
connection_string = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"

conn = pyodbc.connect(connection_string)
print("✅ Connected successfully!")


✅ Connected successfully!


#### Step 6: Load data from a table

Replace YourTableName with one of the table names returned above.

In [ ]:
query = "SELECT * FROM YourTableName"

df = pd.read_sql(query, conn)

print(df.head())
print(df.shape)

In [ ]:
# if want to Close the connection
conn.close()

#### 2.6 For PDF Data

In [ ]:
# Load the PDF
RAW_PDF_PATH = "../data/document.pdf"

text = ""

with pdfplumber.open(RAW_PDF_PATH) as pdf:
    for page in pdf.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"

print(f"✅ PDF loaded: {len(pdf.pages)} pages")
print(text[:1000])  # Preview first 1000 characters

## 3. Understanding Data

In [ ]:
# Display basic info
df.info()

In [ ]:
df.head()  # first 5 rows

In [ ]:
df.tails()  # last 5 rows

## 4.Remove Duplicates

In [ ]:
# Drop duplicate rows if any
df.drop_duplicates(inplace=True)
print(f"✅ After removing duplicates: {df.shape}")

## 5. Handling Missing Values

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0])


In [ ]:
# Visualizing Missing Values Before Cleaning
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cmap="plasma", cbar=False, yticklabels=False)
plt.title("Missing Values Before Cleaning")
plt.show()
#yellow missing values, blue no missing values

In [ ]:
df = df.dropna(axis=1, how="all")  # Drop fully empty columns
df = df.dropna(axis=0, how="all")  # Drop fully empty rows
df = df.dropna(thresh=0.5 * len(df), axis=1)  # Drop columns with >50% missing values
#verify
df.shape
#view
df.head()

In [ ]:
# Visualizing Missing Values after droping columns
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cmap="plasma", cbar=False, yticklabels=False)
plt.title("Missing Values After Droping Columns")
plt.show()

In [ ]:
# To check if any colum dropped
df.info()

In [ ]:
# Fill missing values for **numerical** columns with median
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.median()))

# Fill missing values for **categorical** columns with mode
cat_cols = df.select_dtypes(include=["object"]).columns
df[cat_cols] = df[cat_cols].apply(lambda x: x.fillna(x.mode()[0]))
df.head()

In [ ]:
# Visualizing Missing Values After Cleaning
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cmap="viridis", cbar=False, yticklabels=False)
plt.title("Missing Values After Cleaning")
plt.show()

In [ ]:
#To view changes
df.head()

## 7. Labeling Data

## 8.Encoding Categorical Variables

## 9. Save Processed Data

##### Remember to update file path for each file. Edit: no need to keep track of file name

In [ ]:
PROCESSED_DIR = "../data/datasets/"

# Find the next available filename dynamically
file_number = next(i for i in itertools.count(1) if not os.path.exists(f"{PROCESSED_DIR}final_dataset_{i:02d}.csv"))

# Save the dataset
filename = f"final_dataset_{file_number:02d}.csv"
file_path = os.path.join(PROCESSED_DIR, filename)
#df.to_csv(file_path, index=False)


print(f"✅ Processed dataset saved to {file_path}")
